# Solutions — Nested Data the Spark Way: explode, Structs, Schema Inference, Star Schemas

Exercise notebook: `12_pyspark_preparation/02_nested_data_for_spark.ipynb`. Each solution has: working code, an explanation, complexity, common mistakes, and a Java-style vs Pythonic comparison where it teaches something.

## Exercise 1 — 🟢 Easy · explode vs explode_outer

**Working solution**

In [ ]:
def explode(rows, column, alias, outer=False):
    out = []
    for row in rows:
        base = {k: v for k, v in row.items() if k != column}
        values = row.get(column) or []
        if not values:
            if outer:
                out.append({**base, alias: None})
            continue
        out.extend({**base, alias: v} for v in values)
    return out

**Explanation**

`F.explode` silently drops rows with empty/null arrays — a classic source of 'missing orders' in reports; `explode_outer` keeps them. Know which one you need.

**Complexity:** O(total elements).

**Common mistakes**

- Using `explode` and losing orders that have no items yet.

## Exercise 2 — 🟡 Medium · Flatten structs into columns

**Working solution**

In [ ]:
def flatten_structs(row, sep="_"):
    flat = {}
    def walk(node, prefix):
        for key, value in node.items():
            name = f"{prefix}{sep}{key}" if prefix else key
            if isinstance(value, dict) and value:
                walk(value, name)
            else:
                if name in flat:
                    raise ValueError(f"column name collision: {name}")
                flat[name] = value
    walk(row, "")
    return flat

**Explanation**

Structs flatten cleanly because each has exactly one value per row; arrays don't (they change the row count). Keeping that distinction straight is half of nested-data work.

**Complexity:** O(fields).

**Common mistakes**

- Flattening lists into `items_0_price`, `items_1_price` columns — schema explodes and varies by row.

## Exercise 3 — 🟡 Medium · collect_list / collect_set

**Working solution**

In [ ]:
def collect_by(rows, key, value, distinct=False, sort=False):
    groups = {}
    for r in rows:
        v = r.get(value)
        if v is None:
            continue
        bucket = groups.setdefault(r[key], [])
        if distinct and v in bucket:
            continue
        bucket.append(v)
    if sort:
        groups = {k: sorted(vs) for k, vs in groups.items()}
    return groups

**Explanation**

The inverse of explode: many rows → one row with an array. Note Spark's `collect_set` gives **no ordering guarantee** — sort explicitly (`array_sort`) if order matters.

**Complexity:** O(n × group size) with list membership; use a side set for big groups.

**Common mistakes**

- Relying on the order of `collect_list` after a shuffle.

## Exercise 4 — 🟠 Hard · Schema inference with type merging

**Working solution**

In [ ]:
def _type_of(value):
    if value is None:
        return "null"
    if isinstance(value, bool):
        return "boolean"
    if isinstance(value, int):
        return "long"
    if isinstance(value, float):
        return "double"
    if isinstance(value, str):
        return "string"
    if isinstance(value, dict):
        return {"struct": _merge_structs([{k: _type_of(v) for k, v in value.items()}])}
    if isinstance(value, list):
        element = "null"
        for item in value:
            element = _merge(element, _type_of(item))
        return {"array": element}
    return "string"

def _merge_structs(structs):
    merged = {}
    for s in structs:
        for k, t in s.items():
            merged[k] = _merge(merged.get(k, "null"), t)
    return {k: merged[k] for k in sorted(merged)}

def _merge(a, b):
    if a == "null":
        return b
    if b == "null":
        return a
    if a == b:
        return a
    if isinstance(a, str) and isinstance(b, str) and {a, b} == {"long", "double"}:
        return "double"
    if isinstance(a, dict) and isinstance(b, dict):
        if "struct" in a and "struct" in b:
            return {"struct": _merge_structs([a["struct"], b["struct"]])}
        if "array" in a and "array" in b:
            return {"array": _merge(a["array"], b["array"])}
    return "string"

def infer_schema(records):
    return _merge_structs([{k: _type_of(v) for k, v in r.items()} for r in records])

**Explanation**

This is what `spark.read.json` does in a pass over (a sample of) your data: infer each record's type, then *merge*. Conflicts widen to `string` — which is why production jobs pass an explicit
schema (`.schema(...)`) instead of inferring: inference is slow, sample-dependent, and silently changes types when one producer sends `"12"` instead of `12`.

**Complexity:** O(total values).

**Common mistakes**

- Treating `True` as a long (bool is a subclass of int).
- Inferring from the first record only.

## Challenge — 🔴 Very Hard · Normalize nested orders into a star schema

**Working solution**

In [ ]:
def to_star_schema(orders):
    ordered = sorted(orders, key=lambda o: o["created_at"])
    customers, products = {}, {}
    facts, empty = [], []
    for o in ordered:
        c = o["customer"]
        cust = customers.setdefault(c["id"], {"customer_key": len(customers) + 1, "customer_id": c["id"]})
        cust.update(name=c.get("name"), country=c.get("country"), email=(c.get("contact") or {}).get("email"))
        if not o["items"]:
            empty.append(o["id"])
            continue
        for line_no, item in enumerate(o["items"], start=1):
            prod = products.setdefault(item["product_id"], {"product_key": len(products) + 1, "product_id": item["product_id"],
                                                            "name": item.get("name"), "category": item.get("category") or "UNKNOWN"})
            facts.append({"order_id": o["id"], "line_no": line_no, "customer_key": cust["customer_key"], "product_key": prod["product_key"],
                          "order_date": o["created_at"][:10], "qty": item["qty"], "price": item["price"],
                          "line_total": round(item["qty"] * item["price"], 2),
                          "payment_status": (o.get("payment") or {}).get("status", "unknown")})
    return {"dim_customer": list(customers.values()), "dim_product": list(products.values()),
            "fact_order_items": facts, "fact_orders_without_items": empty}

**Explanation**

Dimensional modelling in miniature: surrogate keys decouple facts from source ids, dimensions are deduplicated with an explicit 'latest wins' rule, facts hold measures + foreign keys at a
declared grain (order line). Keeping a side table for orders without items is how you prove nothing was dropped. In Spark: `explode` + `dropDuplicates`/window + `monotonically_increasing_id`
(or hash keys) + joins.

**Complexity:** O(total items + orders log orders).

**Common mistakes**

- Using the source id as the fact key and losing SCD flexibility later.
- First-seen instead of latest attributes for dimensions.